
# Sentiment Analysis using RNN Variants (SimpleRNN, GRU, LSTM)

This project performs sentiment analysis on the IMDB movie reviews dataset using different RNN architectures: **SimpleRNN**, **GRU**, and **LSTM**.

We preprocess textual data, prepare it for modeling, build and train the models, and evaluate their performance.



## Step 1: Imports and Dataset Loading

We start by importing the necessary libraries and loading the IMDB dataset using Keras.


In [2]:
from tensorflow.keras.layers import SimpleRNN, LSTM, GRU, Bidirectional, Dense, Embedding
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
import numpy as np

In [3]:
# Retrieve reviews with the top 5000 most frequent words
vocab_size = 5000
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)
print(x_train[0])

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 2, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 2, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 2, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 2, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 2, 19, 178, 32]



## Step 2: Mapping Indices to Words

We retrieve the word-to-index dictionary and reverse it to decode reviews back to readable text.


In [4]:
# Retrieve and reverse the word index
word_idx = imdb.get_word_index()
word_idx = {i: word for word, i in word_idx.items()}

# Print the first review in words
print([word_idx.get(i, '?') for i in x_train[0]])

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
['the', 'as', 'you', 'with', 'out', 'themselves', 'powerful', 'lets', 'loves', 'their', 'becomes', 'reaching', 'had', 'journalist', 'of', 'lot', 'from', 'anyone', 'to', 'have', 'after', 'out', 'atmosphere', 'never', 'more', 'room', 'and', 'it', 'so', 'heart', 'shows', 'to', 'years', 'of', 'every', 'never', 'going', 'and', 'help', 'moments', 'or', 'of', 'every', 'chest', 'visual', 'movie', 'except', 'her', 'was', 'several', 'of', 'enough', 'more', 'with', 'is', 'now', 'current', 'film', 'as', 'you', 'of', 'mine', 'potentially', 'unfortunately', 'of', 'you', 'than', 'him', 'that', 'with', 'out', 'themselves', 'her', 'get', 'for', 'was', 'camp', 'of', 'you', 'movie', 'sometimes', 'movie', 'that', 'with', 'scary', 'but', 'and', 'to', 'story', 'wonderful', 'that', 'in', 'seeing', 'in', 'character', 'to', 'of', '70s', 'and', 'with', 'heart', 'had', 'shadows', 'they', 'of', 'here', 'that', 'with', 'her', 'serious', 'to', 'have', 'does', 'when',


## Step 3: Review Length Analysis

We examine the range of review lengths to choose a suitable fixed length.


In [5]:
# Find max and min length of reviews
print("Max length of a review:: ", len(max((x_train+x_test), key=len)))
print("Min length of a review:: ", len(min((x_train+x_test), key=len)))

Max length of a review::  2697
Min length of a review::  70



## Step 4: Padding and Splitting the Dataset

To make all reviews the same length, we pad them to a maximum of 300 words.


In [6]:
from tensorflow.keras.preprocessing import sequence

# Fixing the length of all reviews
max_words = 300
x_train = sequence.pad_sequences(x_train, maxlen=max_words)
x_test = sequence.pad_sequences(x_test, maxlen=max_words)

# Create a validation set
x_valid, y_valid = x_train[:64], y_train[:64]
x_train_, y_train_ = x_train[64:], y_train[64:]


## Step 5: SimpleRNN Model

We define a basic RNN model using `SimpleRNN` layer and train it.


In [7]:
# Word embedding size
embd_len = 32

# Define the model
RNN_model = Sequential(name="Simple_RNN")
RNN_model.add(Embedding(vocab_size, embd_len, input_length=max_words))
RNN_model.add(SimpleRNN(128, activation='tanh', return_sequences=False))
RNN_model.add(Dense(1, activation='sigmoid'))

# Compile the model
RNN_model.compile(loss="binary_crossentropy", optimizer='adam', metrics=['accuracy'])
print(RNN_model.summary())

# Train the model
history = RNN_model.fit(x_train_, y_train_,
                        batch_size=64,
                        epochs=5,
                        verbose=1,
                        validation_data=(x_valid, y_valid))

/Users/riz1ahmed/AI course/AI-NN/.venv/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "Simple_RNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/5
390/390 ━━━━━━━━━━━━━━━━━━━━ 22s 54ms/step - accuracy: 0.5285 - loss: 0.6916 - val_accuracy: 0.6250 - val_loss: 0.6875
Epoch 2/5
390/390 ━━━━━━━━━━━━━━━━━━━━ 21s 53ms/step - accuracy: 0.6752 - loss: 0.5986 - val_accuracy: 0.6875 - val_loss: 0.5847
Epoch 3/5
390/390 ━━━━━━━━━━━━━━━━━━━━ 21s 53ms/step - accuracy: 0.7474 - loss: 0.5196 - val_accuracy: 0.5312 - val_loss: 0.7124
Epoch 4/5
390/390 ━━━━━━━━━━━━━━━━━━━━ 21s 53ms/step - accuracy: 0.7332 - loss: 0.5309 - val_accuracy: 0.7344 - val_loss: 0.5512
Epoch 5/5
390/390 ━━━━━━━━━━━━━━━━━━━━ 21s 53ms/step - accuracy: 0.7815 - loss: 0.4695 - val_accuracy: 0.7969 - val_loss: 0.4761


In [8]:
# Evaluate SimpleRNN
print("Simple_RNN Score---> ", RNN_model.evaluate(x_test, y_test, verbose=0))

Simple_RNN Score--->  [0.46628230810165405, 0.795799970626831]



## Step 6: GRU Model

We now build and train a model using the GRU layer.


In [9]:
# GRU model
gru_model = Sequential(name="GRU_Model")
gru_model.add(Embedding(vocab_size, embd_len, input_length=max_words))
gru_model.add(GRU(128, activation='tanh', return_sequences=False))
gru_model.add(Dense(1, activation='sigmoid'))

# Compile
gru_model.compile(loss="binary_crossentropy", optimizer='adam', metrics=['accuracy'])
print(gru_model.summary())

# Train
history2 = gru_model.fit(x_train_, y_train_,
                         batch_size=64,
                         epochs=5,
                         verbose=1,
                         validation_data=(x_valid, y_valid))

Model: "GRU_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/5
390/390 ━━━━━━━━━━━━━━━━━━━━ 66s 168ms/step - accuracy: 0.7570 - loss: 0.4889 - val_accuracy: 0.9531 - val_loss: 0.2810
Epoch 2/5
390/390 ━━━━━━━━━━━━━━━━━━━━ 67s 171ms/step - accuracy: 0.8763 - loss: 0.3044 - val_accuracy: 0.9531 - val_loss: 0.2202
Epoch 3/5
390/390 ━━━━━━━━━━━━━━━━━━━━ 348s 894ms/step - accuracy: 0.9014 - loss: 0.2480 - val_accuracy: 0.9219 - val_loss: 0.2396
Epoch 4/5
390/390 ━━━━━━━━━━━━━━━━━━━━ 64s 165ms/step - accuracy: 0.9168 - loss: 0.2094 - val_accuracy: 0.9375 - val_loss: 0.1733
Epoch 5/5
390/390 ━━━━━━━━━━━━━━━━━━━━ 127s 326ms/step - accuracy: 0.9403 - loss: 0.1612 - val_accuracy: 0.9219 - val_loss: 0.2161


In [10]:
# Evaluate GRU
print("GRU model Score", gru_model.evaluate(x_test, y_test, verbose=0))

GRU model Score [0.31367018818855286, 0.8849999904632568]



## Step 7: LSTM Model

Finally, we build and train a model using the LSTM layer.


In [11]:
# LSTM model
lstm_model = Sequential(name="LSTM_Model")
lstm_model.add(Embedding(vocab_size, embd_len, input_length=max_words))
lstm_model.add(LSTM(128, activation='relu', return_sequences=False))
lstm_model.add(Dense(1, activation='sigmoid'))

# Compile
lstm_model.compile(loss="binary_crossentropy", optimizer='adam', metrics=['accuracy'])
print(lstm_model.summary())

# Train
history3 = lstm_model.fit(x_train_, y_train_,
                          batch_size=64,
                          epochs=5,
                          verbose=2,
                          validation_data=(x_valid, y_valid))

Model: "LSTM_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/5
390/390 - 61s - 157ms/step - accuracy: 0.5087 - loss: nan - val_accuracy: 0.6094 - val_loss: nan
Epoch 2/5
390/390 - 291s - 745ms/step - accuracy: 0.4997 - loss: nan - val_accuracy: 0.6094 - val_loss: nan
Epoch 3/5
390/390 - 61s - 156ms/step - accuracy: 0.4997 - loss: nan - val_accuracy: 0.6094 - val_loss: nan
Epoch 4/5
390/390 - 61s - 157ms/step - accuracy: 0.4997 - loss: nan - val_accuracy: 0.6094 - val_loss: nan
Epoch 5/5
390/390 - 977s - 3s/step - accuracy: 0.4997 - loss: nan - val_accuracy: 0.6094 - val_loss: nan


In [12]:
# Evaluate LSTM
print("LSTM model Score---> ", lstm_model.evaluate(x_test, y_test, verbose=0))

LSTM model Score--->  [nan, 0.5]


In [ ]:
# Bidirectional LSTM model
bi_lstm_model = Sequential(name="Bi_LSTM_Model")
bi_lstm_model.add(Embedding(vocab_size, embd_len, input_length=max_words))
bi_lstm_model.add(Bidirectional(LSTM(128, activation='relu', return_sequences=False)))
bi_lstm_model.add(Dense(1, activation='sigmoid'))

# Compile
bi_lstm_model.compile(loss="binary_crossentropy", optimizer='adam', metrics=['accuracy'])
print(bi_lstm_model.summary())

#train
history4 = bi_lstm_model.fit(x_train_, y_train_,
                          batch_size=64,
                          epochs=5,
                          verbose=2,
                          validation_data=(x_valid, y_valid))


Model: "Bi_LSTM_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/5
390/390 - 280s - 718ms/step - accuracy: 0.5043 - loss: nan - val_accuracy: 0.6094 - val_loss: nan
Epoch 2/5


In [ ]:
# Evaluate Bi-LSTM
print("Bi-LSTM model Score---> ", bi_lstm_model.evaluate(x_test, y_test, verbose=0))

Bi-LSTM model Score--->  [nan, 0.5]


In [ ]:
# compare the models
print("Simple_RNN Score---> ", RNN_model.evaluate(x_test, y_test, verbose=0))
print("GRU model Score", gru_model.evaluate(x_test, y_test, verbose=0))
print("LSTM model Score---> ", lstm_model.evaluate(x_test, y_test, verbose=0))
print("Bi-LSTM model Score---> ", bi_lstm_model.evaluate(x_test, y_test, verbose=0))

Simple_RNN Score--->  [0.5097653865814209, 0.7553200125694275]
GRU model Score [0.33191293478012085, 0.8724799752235413]
LSTM model Score--->  [nan, 0.5]
Bi-LSTM model Score--->  [nan, 0.5]
